# Day 35 — **httpx: Async HTTP Client for Tool Calls**

**Phase 2 · Module 5: Multi-Agent Orchestration** · ~30 min

Every agent tool that hits an external API — a Java REST service (Day 34), Bedrock, a KYC provider — should use **`httpx.AsyncClient`**, not `requests`. `requests` is blocking: one slow call stalls your whole event loop, killing the `asyncio.gather` parallelism from Day 09. `httpx` is async-native, does connection pooling, and has first-class timeouts.

Everything below runs **offline** using `httpx.MockTransport` — real `httpx` code, faked responses, no network. That's also how you unit-test agent tools (Day 29).

Today:
1. `AsyncClient` + **`MockTransport`** — real client, deterministic responses.
2. **Timeouts** — never wait forever for an external API.
3. **Retry on 429/503** with tenacity + parallel calls with `gather`.
4. **One client for the agent's lifetime** — connection pooling done right.

## 1. AsyncClient with MockTransport

`MockTransport` lets you inject a handler that returns canned `httpx.Response`s — so the *client* is real (same API, timeouts, pooling) but there's no network. Perfect for a notebook and for CI.

In [1]:
import httpx, asyncio, nest_asyncio
nest_asyncio.apply()

def handler(request: httpx.Request) -> httpx.Response:
    """Pretend bank KYC service. Routes on path, echoes JSON."""
    if request.url.path == "/kyc":
        name = request.url.params.get("name", "")
        risk = "HIGH" if name == "Vulpes SA" else "LOW"
        return httpx.Response(200, json={"name": name, "risk": risk})
    return httpx.Response(404, json={"error": "not found"})

transport = httpx.MockTransport(handler)

async def check_kyc(name: str) -> dict:
    async with httpx.AsyncClient(transport=transport, base_url="https://kyc.bank") as client:
        resp = await client.get("/kyc", params={"name": name})
        resp.raise_for_status()
        return resp.json()

print(asyncio.run(check_kyc("ACME Ltd")))
print(asyncio.run(check_kyc("Vulpes SA")))

{'name': 'ACME Ltd', 'risk': 'LOW'}
{'name': 'Vulpes SA', 'risk': 'HIGH'}


☝ `await client.get(...)` is the difference from `requests.get(...)`: it yields the event loop while the socket waits, so other agent tools run concurrently instead of blocking. `raise_for_status()` turns a 4xx/5xx into an exception you can catch — never trust a 200-shaped code path on an error response. In production you drop the `transport=` argument and pass a real `base_url`; the tool body is identical, which is exactly why MockTransport is the right test seam.

## 2. Timeouts — the non-negotiable

An external API that hangs will hang your agent. `httpx.Timeout` caps each phase (connect, read, write, pool). Against a real network, httpx raises `httpx.ReadTimeout` when the server exceeds the read budget; here we make the mock upstream *itself* raise it (a `MockTransport` handler runs synchronously, so it can't be wall-clock-timed) — the point is the **catch-and-degrade** path your tool must have.

In [2]:
# The real config you'd attach to the client: per-phase budgets.
TIMEOUT = httpx.Timeout(connect=1.0, read=2.0, write=1.0, pool=1.0)

def maybe_slow_handler(request: httpx.Request) -> httpx.Response:
    # A real slow server would make httpx raise ReadTimeout; simulate that outcome.
    if request.url.params.get("slow") == "1":
        raise httpx.ReadTimeout("read exceeded 2.0s", request=request)
    return httpx.Response(200, json={"quote": 0.079})

async def get_quote(slow: bool) -> str:
    transport = httpx.MockTransport(maybe_slow_handler)
    async with httpx.AsyncClient(transport=transport, timeout=TIMEOUT) as client:
        try:
            r = await client.get("https://rates.bank/quote", params={"slow": int(slow)})
            return f"got {r.json()}"
        except httpx.ReadTimeout:
            return "TIMED OUT — fell back to cached quote"

print(asyncio.run(get_quote(slow=False)))    # responds in budget -> ok
print(asyncio.run(get_quote(slow=True)))     # exceeds read budget -> ReadTimeout caught

got {'quote': 0.079}
TIMED OUT — fell back to cached quote


☝ A per-phase `Timeout` is more precise than one number: a slow *connect* (DNS/TLS) is a different failure from a slow *read* (server thinking), and you often tune them differently. Catching `httpx.ReadTimeout` is what lets the agent **degrade gracefully** (Day 37) — return a cached quote instead of hanging. Rule for agent tools: no external call without an explicit timeout, ever.

## 3. Retry on 429/503 + parallel calls

Transient throttles deserve a retry (Day 37). `tenacity` wraps the call; `asyncio.gather` fans several out at once. Retry only on *transient* status codes — never on a 400.

In [3]:
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

_attempts = {"n": 0}
def flaky_handler(request: httpx.Request) -> httpx.Response:
    _attempts["n"] += 1
    if _attempts["n"] < 3:
        return httpx.Response(503, json={"error": "unavailable"})   # transient
    return httpx.Response(200, json={"rate": 0.079})

class Transient(Exception): pass

@retry(retry=retry_if_exception_type(Transient),
       stop=stop_after_attempt(4), wait=wait_exponential(multiplier=0.05, max=1))
async def get_rate(client: httpx.AsyncClient) -> dict:
    r = await client.get("https://rates.bank/base")
    if r.status_code in (429, 503):
        raise Transient(f"{r.status_code} — retrying")
    r.raise_for_status()
    return r.json()

async def main():
    transport = httpx.MockTransport(flaky_handler)
    async with httpx.AsyncClient(transport=transport) as client:
        return await get_rate(client)

print("after retries:", asyncio.run(main()), "| total attempts:", _attempts["n"])

after retries: {'rate': 0.079} | total attempts: 3


☝ `retry_if_exception_type(Transient)` is the guardrail: a 503 raises `Transient` and retries with exponential backoff, but a `raise_for_status()` 400 propagates immediately — retrying a malformed request just wastes attempts. Pair this with `asyncio.gather(*[get_rate(c) for ...])` to run many tool calls in parallel, each independently retried. This is the composition that makes agent tool-calling both fast *and* resilient.

## 4. One client for the agent's lifetime

Creating an `AsyncClient` per call throws away the connection pool — you pay TLS handshake every time. Create **one** client when the agent starts, reuse it, close it on shutdown. `async with` is fine for a script; a long-lived agent holds the client as an attribute.

In [4]:
class AgentHttp:
    """Owns ONE pooled client for the whole agent lifetime."""
    def __init__(self, transport=None):
        self._client = httpx.AsyncClient(transport=transport, base_url="https://kyc.bank",
                                         limits=httpx.Limits(max_connections=20))
    async def kyc(self, name: str) -> dict:
        r = await self._client.get("/kyc", params={"name": name})
        r.raise_for_status(); return r.json()
    async def aclose(self):
        await self._client.aclose()

async def lifetime_demo():
    http = AgentHttp(transport=httpx.MockTransport(handler))
    try:
        results = await asyncio.gather(http.kyc("ACME Ltd"), http.kyc("Vulpes SA"))
        return results
    finally:
        await http.aclose()                            # release the pool on shutdown

print(asyncio.run(lifetime_demo()))

[{'name': 'ACME Ltd', 'risk': 'LOW'}, {'name': 'Vulpes SA', 'risk': 'HIGH'}]


☝ `httpx.Limits(max_connections=20)` caps the pool so one busy agent can't exhaust file descriptors — the client-side complement to the `asyncio.Semaphore` rate-limiting from Day 15. The two parallel `kyc` calls **reuse** pooled connections instead of handshaking twice. And `aclose()` in a `finally` mirrors the async-context-manager discipline from Day 06: a leaked client leaks sockets. One client, pooled, closed on shutdown — that's the production shape.

## Recap

| Concern | httpx mechanism |
|---|---|
| Don't block the event loop | `AsyncClient` + `await` |
| Test offline / in CI | `MockTransport` (real client, fake responses) |
| Never hang | `httpx.Timeout(connect/read/write/pool)` |
| Survive throttles | tenacity retry on 429/503 only |
| Run tools in parallel | `asyncio.gather` |
| Reuse connections | one client for the agent's life + `Limits` |

**Rule:** `requests` for a quick script; `httpx.AsyncClient` for anything an agent calls. Tomorrow (Day 36): Redis as the shared-memory store behind a multi-agent system.